In [ ]:
%load_ext autoreload
%autoreload 2


# Flight Delay ML Platform — Historical Feature Engineering

## 1. Point-in-Time Feature Availability Contract

### Prediction Objective

Predict departure and arrival delays using information that would be available 24 hours before scheduled departure.

### Current Data Limitation

The baseline uses retrospective BTS scheduled-flight records as a proxy for pre-departure information.

Historical BTS delay outcomes are published after flight operations and cannot automatically be treated as available immediately after a flight.

### Availability Rule

A historical record may contribute to a prediction feature only if:

1. The source data was available before the prediction timestamp.
2. The source version used can be identified.
3. The record does not include outcomes from the target flight.
4. The feature calculation does not include observations unavailable at the prediction timestamp.

### Initial Historical Features

- Origin departure delay rate
- Origin arrival delay rate
- Airline-origin departure delay rate
- Airline-origin arrival delay rate

Each feature will use up to three previously available reporting months.

### Missing History

If insufficient historical observations are available, the pipeline will use an explicitly defined fallback or missing-value strategy.

### Data Availability Modes

**Strict mode:** Requires evidence of source availability and an identifiable source snapshot.

**Research proxy mode:** Uses a documented reporting-delay assumption. Results must be labeled as retrospective simulations rather than verified real-time predictions.

### Leakage Prevention

Training and validation features must be constructed using the same point-in-time rules.

The 2025 test set remains locked until final evaluation.

The current modeling cohort includes only completed, non-diverted flights with both delay labels available. Predictions therefore describe delay risk conditional on this retrospective cohort, not unconditional risk for every scheduled flight.


In [ ]:
import pandas as pd

# Illustrative source availability dates.
# These are synthetic test timestamps,
# NOT verified BTS publication timestamps.

source_manifest = pd.DataFrame({
    "data_month": [
        "2023-10",
        "2023-12",
        "2024-01",
    ],
    "available_at": pd.to_datetime(
        [
            "2024-01-06T12:00:00Z",
            "2024-03-02T12:00:00Z",
            "2024-04-12T12:00:00Z",
        ],
        utc=True,
    ),
    "snapshot_id": [
        "example_oct",
        "example_dec",
        "example_jan",
    ],
})

# Predict a flight on February 10, 2024,
# using a hypothetical T-24h timestamp.
prediction_at = pd.Timestamp(
    "2024-02-09T18:00:00Z"
)

eligible_history = source_manifest.loc[
    source_manifest["available_at"] <= prediction_at
].copy()

print("=== Point-in-Time Availability Check ===")
print("Prediction timestamp:", prediction_at)

display(eligible_history)

assert eligible_history["data_month"].tolist() == [
    "2023-10"
]

assert (
    eligible_history["available_at"]
    <= prediction_at
).all()

print("\nPoint-in-time availability check passed!")


## 2. Historical Source Availability Calendar

### Research Proxy Policy

Historical BTS source availability is approximated using a fixed 120-day lag after the end of each reporting month.

This is an experimental assumption, not a verified BTS publication schedule.

For each source month, the calendar records:

- Reporting month
- Expected BTS source filename
- Assumed availability timestamp (UTC)
- Availability policy
- Evidence status
- Assumed reporting lag

### Point-in-Time Selection

For each prediction timestamp, only source months with assumed availability timestamps at or before the prediction cutoff are eligible.

The initial historical feature window uses up to three most recently eligible reporting months.

The calculation must never use source months that become available after the prediction cutoff.

### Limitations

The proxy does not establish historical source-version availability. A currently downloaded BTS archive may contain revisions that were not present in earlier releases.

Therefore, results using this policy will remain labeled as retrospective research simulations.


In [ ]:
import pandas as pd

from flight_delay.features.availability import (
    build_proxy_availability_calendar,
    eligible_source_months,
)

# Create a calendar for 2023–2024 source months.
# Do not use 2025 test outcomes during development.
source_months = [
    f"{year}-{month:02d}"
    for year in (2023, 2024)
    for month in range(1, 13)
]

calendar = build_proxy_availability_calendar(
    months=source_months,
    lag_days=120,
)

print("=== Source Availability Calendar ===")
print("Source months:", len(calendar))
print("Policy:", calendar["availability_policy"].unique())

display(calendar.head(5))

# Example prediction timestamp.
prediction_at = pd.Timestamp(
    "2024-02-09T18:00:00Z"
)

eligible = eligible_source_months(
    calendar,
    prediction_at_utc=prediction_at,
    lookback_months=3,
)

print("\n=== Eligible Historical Months ===")
display(
    eligible[
        [
            "report_month",
            "assumed_available_at_utc",
            "availability_evidence",
        ]
    ]
)

assert len(calendar) == 24

assert eligible["report_month"].tolist() == [
    "2023-07",
    "2023-08",
    "2023-09",
]

assert (
    eligible["assumed_available_at_utc"]
    <= prediction_at
).all()

assert (
    eligible["availability_evidence"]
    == "assumed_not_verified"
).all()

print("\nHistorical availability calendar passed!")

In [ ]:
from pathlib import Path

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)

print("Project root:", ROOT)

In [ ]:
import json
from pathlib import Path

# ROOT should already point to the project root.
manifest_files = sorted(
    (ROOT / "data").rglob("*.json")
)

print("=== Existing JSON Files ===")
print("Total:", len(manifest_files))

for path in manifest_files[:10]:
    print(path.relative_to(ROOT))

print("\n=== Sample Manifest Structures ===")

for path in manifest_files[:3]:
    with path.open(encoding="utf-8") as f:
        content = json.load(f)

    print(f"\nFile: {path.name}")

    if isinstance(content, dict):
        print("Top-level keys:", list(content.keys()))
        for key, value in content.items():
            print(f"  {key}: {type(value).__name__}")
    else:
        print("Root type:", type(content).__name__)

In [ ]:
import json

manifest_dir = ROOT / "data" / "interim" / "manifests"

manifest_files = sorted(
    manifest_dir.glob("????-??.json")
)

expected_months = {
    "2022-06",
    "2022-07",
    "2022-08",
    "2022-09",
    "2022-10",
    "2022-11",
    "2022-12",
} | {
    f"{year}-{month:02d}"
    for year in range(2023, 2026)
    for month in range(1, 13)
}

actual_months = {
    path.stem for path in manifest_files
}

missing_months = expected_months - actual_months
unexpected_months = actual_months - expected_months

print("=== Monthly Manifest Files ===")
print("Total monthly manifests:", len(manifest_files))
print("Expected months:", len(expected_months))
print("Missing months:", sorted(missing_months))
print("Unexpected months:", sorted(unexpected_months))

assert not missing_months, "Missing monthly manifests"
assert not unexpected_months, "Unexpected monthly manifests"

print("\nMonthly manifest coverage passed!")

# Inspect one actual monthly manifest.
example_path = manifest_dir / "2023-01.json"

with example_path.open(encoding="utf-8") as f:
    manifest = json.load(f)

print("\n=== January 2023 Manifest ===")
print("File:", example_path.relative_to(ROOT))

print("\nTop-level keys:")
for key, value in manifest.items():
    print(f"{key}: {type(value).__name__}")

print("\n=== Full Manifest Content ===")
print(json.dumps(manifest, indent=2))


## 3. Historical Source Provenance Integration

### Objective

Integrate existing monthly BTS processing manifests with the research-proxy availability calendar.

### Data Lineage

Each historical source month is associated with:

- Reporting month
- Raw ZIP filename and SHA256
- Processed Parquet filename and SHA256
- Processed row count
- Schema fingerprint
- Dataset version
- Manifest generation timestamp
- Assumed source availability timestamp

### Development Scope

Only source months from January 2023 through August 2024 are included in this initial integration.

The calibration, threshold-selection, and final test periods remain outside this development-stage source inventory.

### Important Distinction

The SHA256 digests identify the files currently used in the project.

The 120-day availability lag is a research assumption, not evidence that the same source version was publicly available at the historical prediction timestamp.

Therefore, this integration provides reproducible source lineage but does not establish strict point-in-time source-version validity.


In [ ]:
from flight_delay.features.source_provenance import (
    load_source_provenance,
)

# Keep development-stage sources separate from
# calibration, threshold-selection, and test periods.
development_calendar = calendar.loc[
    calendar["report_month"] <= "2024-08"
].copy()

provenance = load_source_provenance(
    manifest_dir=ROOT / "data/interim/manifests",
    availability_calendar=development_calendar,
)

print("=== Historical Source Provenance ===")
print("Source months:", len(provenance))
print(
    "Dataset versions:",
    provenance["dataset_version"].unique(),
)
print(
    "Schema fingerprints:",
    provenance["schema_sha256"].nunique(),
)
print(
    "Availability policies:",
    provenance["availability_policy"].unique(),
)

display(
    provenance[
        [
            "report_month",
            "processed_filename",
            "processed_row_count",
            "assumed_available_at_utc",
            "availability_evidence",
        ]
    ].head(8)
)

assert len(provenance) == 20

assert provenance["report_month"].iloc[0] == "2023-01"
assert provenance["report_month"].iloc[-1] == "2024-08"

assert provenance["schema_sha256"].nunique() == 1

assert (
    provenance["availability_evidence"]
    == "assumed_not_verified"
).all()

assert not provenance["report_month"].str.startswith(
    "2025"
).any()

print("\nSource provenance integration passed!")


## 4. Source File Integrity Verification

### Objective

Verify that the actual source ZIP and processed Parquet files match the SHA256 values recorded in their monthly manifests.

### Method

- Locate the files referenced by the provenance table.
- Calculate SHA256 directly from file bytes.
- Compare calculated hashes with manifest values.
- Fail if a located file differs from its recorded hash.

### Scope

January 2023 is used as an initial integrity verification.

A successful result proves consistency with the current manifest, not historical availability at the original prediction timestamp.


In [ ]:
import hashlib
from pathlib import Path


def calculate_sha256(path: Path) -> str:
    """Calculate a file SHA256 without loading it all into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


# Select January 2023 provenance.
jan_source = provenance.loc[
    provenance["report_month"] == "2023-01"
].iloc[0]


def find_unique_file(directory: Path, filename: str):
    matches = list(directory.rglob(filename))

    if len(matches) != 1:
        print(
            f"Expected one file named {filename}, "
            f"found {len(matches)}"
        )
        return None

    return matches[0]


# Find actual source ZIP and processed Parquet.
source_path = find_unique_file(
    ROOT / "data/raw",
    jan_source["source_filename"],
)

processed_path = find_unique_file(
    ROOT / "data/processed",
    jan_source["processed_filename"],
)

checks = [
    ("Source ZIP", source_path, jan_source["source_sha256"]),
    (
        "Processed Parquet",
        processed_path,
        jan_source["processed_sha256"],
    ),
]

print("=== January 2023 File Integrity ===")

for name, path, expected_hash in checks:

    if path is None:
        print(f"{name}: NOT VERIFIED — file not located")
        continue

    actual_hash = calculate_sha256(path)
    matches = actual_hash.lower() == expected_hash.lower()

    print(f"\n{name}")
    print("File:", path.relative_to(ROOT))
    print("SHA256 matches manifest:", matches)

    assert matches, f"{name} SHA256 mismatch!"

print("\nFile integrity inspection completed!")


## 5. Historical Feature Cold-Start Analysis

### Problem

The model training period begins in January 2023, but historical delay statistics require source records that were available before each prediction timestamp.

Under the 120-day research-proxy availability policy, 2023 BTS records cannot provide historical statistics for the beginning of 2023.

### Proposed Solution

Use June–December 2022 BTS records as historical feature sources.

These records will only contribute to historical feature aggregation. They will not be included as model training targets.

### Dataset Roles

- 2022-06 through 2022-12: Historical feature source
- 2023: Model training
- January–August 2024: Model validation
- September–October 2024: Calibration
- November–December 2024: Threshold selection
- 2025: Final test

### Leakage Control

Historical source months must satisfy the point-in-time availability policy before contributing to a flight's features.

Availability timestamps in the research-proxy mode are assumed rather than historically verified.


In [ ]:
from flight_delay.features.availability import (
    build_proxy_availability_calendar,
    eligible_source_months,
)

# Candidate historical sources.
source_months_extended = [
    f"{year}-{month:02d}"
    for year in [2022, 2023]
    for month in range(1, 13)
]

extended_calendar = build_proxy_availability_calendar(
    source_months_extended,
    lag_days=120,
)

# Illustrative UTC cutoffs, not actual per-flight T-24h.
prediction_examples = {
    "2023-01-01": "2023-01-01T00:00:00Z",
    "2023-05-01": "2023-05-01T00:00:00Z",
    "2023-06-01": "2023-06-01T00:00:00Z",
}

for label, cutoff in prediction_examples.items():
    eligible = eligible_source_months(
        extended_calendar,
        prediction_at_utc=cutoff,
        lookback_months=3,
    )

    print(f"\nPrediction cutoff: {label}")
    print(
        "Historical months:",
        eligible["report_month"].tolist(),
    )

# Verify earliest training-period historical window.
first_window = eligible_source_months(
    extended_calendar,
    "2023-01-01T00:00:00Z",
    lookback_months=3,
)

assert first_window["report_month"].tolist() == [
    "2022-06",
    "2022-07",
    "2022-08",
]

print("\nHistorical cold-start analysis passed!")


## 6. Historical Source Data Ingestion

### Objective

Ingest June–December 2022 BTS records to provide historical context for the beginning of the 2023 training period.

### Processing Pipeline

The existing BTS ingestion pipeline is reused for:

1. Source ZIP downloading
2. Raw data validation
3. Modeling cohort construction
4. Parquet storage
5. Monthly provenance manifest generation

### Dataset Role

2022 records are historical feature sources only. They are not included in the model training, validation, calibration, threshold-selection, or final test targets.

### Quality Checks

Verify that:

- All seven historical months are present.
- Each month has a processed Parquet file.
- Each month has a source provenance manifest.
- All historical months use a consistent dataset version and schema.
- The historical schema matches the existing 2023 modeling data.

These checks establish structural compatibility but do not verify historical publication timestamps.


In [ ]:
import json
from pathlib import Path

import pandas as pd

historical_months = [
    f"2022-{month:02d}"
    for month in range(6, 13)
]

records = []

for month in historical_months:
    manifest_path = (
        ROOT
        / "data"
        / "interim"
        / "manifests"
        / f"{month}.json"
    )

    assert manifest_path.is_file(), (
        f"Missing manifest: {month}"
    )

    with manifest_path.open(encoding="utf-8") as f:
        manifest = json.load(f)

    processed_path = (
        ROOT
        / "data"
        / "processed"
        / manifest["processed"]["filename"]
    )

    assert processed_path.is_file(), (
        f"Missing Parquet: {month}"
    )

    assert manifest["period"] == month

    records.append({
        "Month": month,
        "Rows": manifest["processed"]["row_count"],
        "DatasetVersion": manifest["dataset_version"],
        "SchemaSHA256": manifest["schema"]["sha256"],
        "ProcessedFile": processed_path.name,
    })

historical_audit = pd.DataFrame(records)

print("=== 2022 Historical Source Audit ===")

display(historical_audit)

print("\nTotal historical months:", len(historical_audit))
print("Total eligible records:", historical_audit["Rows"].sum())
print(
    "Unique schema fingerprints:",
    historical_audit["SchemaSHA256"].nunique(),
)

assert len(historical_audit) == 7
assert historical_audit["Rows"].gt(0).all()
assert historical_audit["SchemaSHA256"].nunique() == 1
assert historical_audit["DatasetVersion"].nunique() == 1

# Confirm compatibility with 2023 training data.
with (
    ROOT / "data/interim/manifests/2023-01.json"
).open(encoding="utf-8") as f:
    train_manifest = json.load(f)

assert (
    historical_audit["SchemaSHA256"].iloc[0]
    == train_manifest["schema"]["sha256"]
), "Historical and training schemas differ!"

print("\n2022 historical source validation passed!")


## 7. Extended Historical Source Provenance

### Objective

Extend the historical source inventory to include June–December 2022 records and resolve the cold-start problem at the beginning of the 2023 training period.

### Source Coverage

- Start: June 2022
- End: August 2024
- Total source months: 27

### Availability Policy

All source months follow the same 120-day research-proxy availability rule.

Inclusion in the source inventory does not imply eligibility for a particular prediction.

Each source month must satisfy the prediction-specific availability cutoff before contributing to historical features.

### Integrity Verification

The actual source ZIP and processed Parquet files will be checked against the SHA256 digests and file sizes recorded in their monthly manifests.

The 2025 test data remains excluded from development-stage feature construction.


In [ ]:
import pandas as pd

from flight_delay.features.availability import (
    build_proxy_availability_calendar,
    eligible_source_months,
)

from flight_delay.features.source_provenance import (
    load_source_provenance,
)

# Historical source inventory for development.
historical_source_months = (
    pd.period_range(
        start="2022-06",
        end="2024-08",
        freq="M",
    )
    .astype(str)
    .tolist()
)

# Build the extended availability calendar.
historical_calendar = build_proxy_availability_calendar(
    historical_source_months,
    lag_days=120,
)

# Integrate all 27 monthly manifests.
historical_provenance = load_source_provenance(
    manifest_dir=ROOT / "data/interim/manifests",
    availability_calendar=historical_calendar,
)

print("=== Extended Historical Source Provenance ===")

print("Source months:", len(historical_provenance))
print(
    "First source month:",
    historical_provenance["report_month"].min(),
)
print(
    "Last source month:",
    historical_provenance["report_month"].max(),
)
print(
    "Total source records:",
    historical_provenance["processed_row_count"].sum(),
)
print(
    "Schema fingerprints:",
    historical_provenance["schema_sha256"].nunique(),
)

assert len(historical_provenance) == 27

assert (
    historical_provenance["report_month"].tolist()
    == historical_source_months
)

assert historical_provenance["schema_sha256"].nunique() == 1
assert historical_provenance["dataset_version"].nunique() == 1

assert (
    historical_provenance["availability_evidence"]
    == "assumed_not_verified"
).all()

display(
    historical_provenance[
        [
            "report_month",
            "processed_row_count",
            "assumed_available_at_utc",
        ]
    ].head(10)
)

print("\nExtended source provenance passed!")

In [ ]:
from flight_delay.data.manifest import sha256_file

integrity_results = []

for row in historical_provenance.itertuples(index=False):

    source_path = (
        ROOT / "data/raw" / row.source_filename
    )

    processed_path = (
        ROOT / "data/processed" / row.processed_filename
    )

    assert source_path.is_file(), (
        f"Missing source: {source_path}"
    )

    assert processed_path.is_file(), (
        f"Missing processed file: {processed_path}"
    )

    source_match = (
        sha256_file(source_path).lower()
        == row.source_sha256.lower()
    )

    processed_match = (
        sha256_file(processed_path).lower()
        == row.processed_sha256.lower()
    )

    source_size_match = (
        source_path.stat().st_size
        == row.source_size_bytes
    )

    processed_size_match = (
        processed_path.stat().st_size
        == row.processed_size_bytes
    )

    integrity_results.append({
        "Month": row.report_month,
        "SourceSHA256Match": source_match,
        "ProcessedSHA256Match": processed_match,
        "SourceSizeMatch": source_size_match,
        "ProcessedSizeMatch": processed_size_match,
    })

integrity_report = pd.DataFrame(integrity_results)

check_columns = [
    "SourceSHA256Match",
    "ProcessedSHA256Match",
    "SourceSizeMatch",
    "ProcessedSizeMatch",
]

display(integrity_report)

assert len(integrity_report) == 27

assert integrity_report[check_columns].all().all(), (
    "Historical source integrity verification failed"
)

print("\n=== Historical Source Integrity Summary ===")
print("Months verified:", len(integrity_report))
print("Files verified:", len(integrity_report) * 2)
print("All hashes match: True")
print("All file sizes match: True")

print("\nFull historical source integrity passed!")


## 8. Monthly Historical Delay Aggregation

### Objective

Construct compact monthly historical statistics for origin airports and airline-origin combinations.

### Aggregation Levels

1. Reporting month × Origin
2. Reporting month × Origin × Reporting Airline

### Aggregated Statistics

- Total eligible flight count
- Departure delay count
- Arrival delay count

Raw counts are retained instead of averaging monthly delay rates. This allows future rolling historical rates to be calculated using the correct observation-weighted denominator.

### Cohort Definition

Only completed, non-diverted flights with valid departure and arrival delay labels are included.

Origin-level arrival delay statistics describe the arrival outcomes of flights departing from that origin airport, rather than all flights arriving at that airport.

### Leakage Control

Monthly aggregation is an offline preparation step.

These statistics must not be joined directly to target flights without applying the prediction-specific source availability cutoff.

The 120-day availability policy remains a retrospective research assumption.


In [ ]:
from flight_delay.data.storage import load_parquet

from flight_delay.features.historical_aggregation import (
    aggregate_monthly_delay_counts,
)

jan_raw = load_parquet(
    ROOT / "data/processed/bts_2023_01_completed.parquet"
)

jan_origin_stats, jan_airline_origin_stats = (
    aggregate_monthly_delay_counts(
        jan_raw,
        report_month="2023-01",
    )
)

print("=== January 2023 Origin Statistics ===")
display(jan_origin_stats)

print("\n=== January 2023 Airline-Origin Statistics ===")
display(jan_airline_origin_stats.head(15))

print("\n=== Aggregation Summary ===")

print("Source records:", len(jan_raw))
print("Origin groups:", len(jan_origin_stats))
print("Airline-origin groups:", len(jan_airline_origin_stats))

assert jan_origin_stats["flight_count"].sum() == 112_548
assert jan_airline_origin_stats["flight_count"].sum() == 112_548

assert len(jan_origin_stats) == 6

print("\nMonthly aggregation pilot passed!")

In [ ]:
import pandas as pd

# Ensure source integrity was checked successfully.
assert len(integrity_report) == 27

assert set(integrity_report["Month"]) == set(
    historical_provenance["report_month"]
)

assert integrity_report[check_columns].all().all()

origin_tables = []
airline_origin_tables = []

for row in historical_provenance.itertuples(index=False):

    path = (
        ROOT
        / "data/processed"
        / row.processed_filename
    )

    monthly_data = load_parquet(path)

    # Reconcile with the existing manifest.
    assert len(monthly_data) == row.processed_row_count

    origin_stats, airline_origin_stats = (
        aggregate_monthly_delay_counts(
            monthly_data,
            report_month=row.report_month,
        )
    )

    origin_tables.append(origin_stats)
    airline_origin_tables.append(airline_origin_stats)

    print(
        f"{row.report_month} | "
        f"flights={len(monthly_data):,} | "
        f"origin_groups={len(origin_stats)} | "
        f"airline_origin_groups={len(airline_origin_stats)}"
    )

    del monthly_data

# Combine aggregated tables, not raw flight records.
historical_origin_stats = pd.concat(
    origin_tables,
    ignore_index=True,
)

historical_airline_origin_stats = pd.concat(
    airline_origin_tables,
    ignore_index=True,
)

print("\n=== Historical Aggregation Summary ===")

print(
    "Origin statistics shape:",
    historical_origin_stats.shape,
)

print(
    "Airline-origin statistics shape:",
    historical_airline_origin_stats.shape,
)

print(
    "Total source flights:",
    historical_origin_stats["flight_count"].sum(),
)

display(historical_origin_stats.head(10))
display(historical_airline_origin_stats.head(10))

In [ ]:
from flight_delay.features.historical_aggregation import (
    COUNT_COLUMNS,
)

expected_records = historical_provenance[
    "processed_row_count"
].sum()

# Check coverage and total counts.
for stats in [
    historical_origin_stats,
    historical_airline_origin_stats,
]:

    assert stats["report_month"].nunique() == 27

    assert set(stats["report_month"]) == set(
        historical_provenance["report_month"]
    )

    assert stats["flight_count"].sum() == expected_records

    assert stats["flight_count"].gt(0).all()

    assert (
        stats[list(COUNT_COLUMNS)] >= 0
    ).all().all()

    assert (
        stats["dep_delay_count"] <= stats["flight_count"]
    ).all()

    assert (
        stats["arr_delay_count"] <= stats["flight_count"]
    ).all()

# Check unique aggregation keys.
assert not historical_origin_stats.duplicated(
    ["report_month", "Origin"]
).any()

assert not historical_airline_origin_stats.duplicated(
    ["report_month", "Origin", "Reporting_Airline"]
).any()

# Reconcile airline-origin counts with origin counts.
reaggregated_origin = (
    historical_airline_origin_stats
    .groupby(["report_month", "Origin"])[list(COUNT_COLUMNS)]
    .sum()
    .sort_index()
)

expected_origin = (
    historical_origin_stats
    .set_index(["report_month", "Origin"])[list(COUNT_COLUMNS)]
    .sort_index()
)

pd.testing.assert_frame_equal(
    reaggregated_origin,
    expected_origin,
    check_dtype=False,
)

print("=== Historical Aggregation Quality Checks ===")
print("Source months: 27")
print("Total source flights:", f"{expected_records:,}")
print("Duplicate aggregation keys: 0")
print("Invalid delay counts: 0")
print("Origin reconciliation: Passed")

print("\nHistorical aggregation validation passed!")


## 9. Prediction Timestamp Construction

### Objective

Construct a prediction timestamp 24 hours before scheduled departure for each eligible flight.

### Input Columns

- `FlightDate`
- `Origin`
- `CRSDepTime`

### Timezone Handling

Scheduled departure times are interpreted using the local timezone of the origin airport.

Timezone-aware timestamps are converted to UTC before subtracting the 24-hour prediction horizon.

### Exceptional Cases

Ambiguous and nonexistent local times during daylight saving transitions are explicitly flagged.

Special scheduled time values of 2400 are also flagged until their date interpretation is resolved.

No target flights are silently dropped.

### Limitations

Scheduled departure times are obtained from retrospective BTS records.

Therefore, the constructed timestamps are retrospective prediction-time proxies and do not establish that the same scheduled departure information was available 24 hours before the actual flight.


In [ ]:
import pandas as pd

from flight_delay.features.prediction_time import (
    build_prediction_timestamps,
)

timestamp_examples = pd.DataFrame({
    "FlightDate": [
        "2024-01-15",
        "2024-01-15",
        "2024-01-15",
        "2024-01-15",
        "2024-07-15",
        "2024-03-10",
        "2024-11-03",
        "2024-01-15",
    ],
    "Origin": [
        "ATL",
        "DFW",
        "DEN",
        "SFO",
        "SFO",
        "SFO",
        "SFO",
        "SFO",
    ],
    "CRSDepTime": [
        800,
        800,
        800,
        800,
        800,
        230,
        130,
        2400,
    ],
})

example_timestamps = build_prediction_timestamps(
    timestamp_examples
)

display(
    pd.concat(
        [timestamp_examples, example_timestamps],
        axis=1,
    )
)

# Verify winter timezone conversions.
assert example_timestamps.loc[
    0, "scheduled_departure_utc"
] == pd.Timestamp("2024-01-15T13:00:00Z")

assert example_timestamps.loc[
    3, "scheduled_departure_utc"
] == pd.Timestamp("2024-01-15T16:00:00Z")

# Verify summer timezone conversion.
assert example_timestamps.loc[
    4, "scheduled_departure_utc"
] == pd.Timestamp("2024-07-15T15:00:00Z")

# Verify DST exceptions and unresolved 2400.
assert example_timestamps.loc[
    5:6, "prediction_at_utc"
].isna().all()

assert example_timestamps.loc[
    7, "timestamp_status"
] == "unresolved_2400"

# Verify exact 24-hour prediction horizon.
valid = example_timestamps["timestamp_status"].eq("ok")

horizon = (
    example_timestamps.loc[
        valid, "scheduled_departure_utc"
    ]
    - example_timestamps.loc[
        valid, "prediction_at_utc"
    ]
)

assert horizon.eq(pd.Timedelta(hours=24)).all()

print("\nSynthetic prediction timestamp checks passed!")

In [ ]:
jan_prediction_times = build_prediction_timestamps(
    jan_raw
)

print("=== January 2023 Prediction Timestamp Audit ===")

print("Source records:", len(jan_raw))
print("Timestamp records:", len(jan_prediction_times))

print("\nTimestamp status:")
display(
    jan_prediction_times[
        "timestamp_status"
    ].value_counts(dropna=False)
)

print("\nEarliest prediction cutoff:")
print(
    jan_prediction_times[
        "prediction_at_utc"
    ].min()
)

print("\nLatest prediction cutoff:")
print(
    jan_prediction_times[
        "prediction_at_utc"
    ].max()
)

valid = jan_prediction_times[
    "timestamp_status"
].eq("ok")

assert len(jan_prediction_times) == len(jan_raw)

assert jan_prediction_times.loc[
    valid, "prediction_at_utc"
].notna().all()

assert jan_prediction_times.loc[
    ~valid, "prediction_at_utc"
].isna().all()

horizon = (
    jan_prediction_times.loc[
        valid, "scheduled_departure_utc"
    ]
    - jan_prediction_times.loc[
        valid, "prediction_at_utc"
    ]
)

assert horizon.eq(pd.Timedelta(hours=24)).all()

display(
    pd.concat(
        [
            jan_raw[
                ["FlightDate", "Origin", "CRSDepTime"]
            ],
            jan_prediction_times,
        ],
        axis=1,
    ).head(10)
)

print("\nJanuary prediction timestamp audit passed!")


## 10. Full Prediction Timestamp Audit

### Objective

Validate prediction timestamp construction across the complete training and validation periods.

### Scope

- Training: January–December 2023
- Validation: January–August 2024

### Validation Checks

- Preserve the number and order of flight records.
- Verify the exact 24-hour prediction horizon.
- Identify unresolved daylight saving timestamps.
- Identify unresolved scheduled departure times of 2400.
- Measure monthly timestamp coverage.
- Verify historical source coverage at the earliest prediction cutoff.

No unresolved records are silently removed.

The calibration, threshold-selection, and final test datasets remain excluded from this development-stage audit.


In [ ]:
import pandas as pd

from flight_delay.data.storage import load_parquet
from flight_delay.features.prediction_time import (
    build_prediction_timestamps,
)

target_inventory = historical_provenance.loc[
    historical_provenance["report_month"].between(
        "2023-01", "2024-08"
    )
].copy()

assert len(target_inventory) == 20

timestamp_audit_results = []

for row in target_inventory.itertuples(index=False):

    path = (
        ROOT
        / "data/processed"
        / row.processed_filename
    )

    monthly_data = load_parquet(path)

    assert len(monthly_data) == row.processed_row_count

    timestamps = build_prediction_timestamps(
        monthly_data
    )

    valid = timestamps["timestamp_status"].eq("ok")

    unresolved_dst = timestamps[
        "timestamp_status"
    ].eq("unresolved_dst")

    unresolved_2400 = timestamps[
        "timestamp_status"
    ].eq("unresolved_2400")

    # Verify record alignment.
    assert len(timestamps) == len(monthly_data)
    assert timestamps.index.equals(monthly_data.index)

    # Verify valid and unresolved timestamps.
    assert timestamps.loc[
        valid, "prediction_at_utc"
    ].notna().all()

    assert timestamps.loc[
        ~valid, "prediction_at_utc"
    ].isna().all()

    # Verify exact prediction horizon.
    horizon = (
        timestamps.loc[
            valid, "scheduled_departure_utc"
        ]
        - timestamps.loc[
            valid, "prediction_at_utc"
        ]
    )

    assert horizon.eq(
        pd.Timedelta(hours=24)
    ).all()

    assert (
        valid.sum()
        + unresolved_dst.sum()
        + unresolved_2400.sum()
        == len(monthly_data)
    )

    timestamp_audit_results.append({
        "Month": row.report_month,
        "Records": len(monthly_data),
        "ValidTimestamps": int(valid.sum()),
        "UnresolvedDST": int(unresolved_dst.sum()),
        "Unresolved2400": int(unresolved_2400.sum()),
        "Coverage": float(valid.mean()),
        "EarliestPredictionUTC": timestamps[
            "prediction_at_utc"
        ].min(),
        "LatestPredictionUTC": timestamps[
            "prediction_at_utc"
        ].max(),
    })

    print(
        f"{row.report_month} | "
        f"records={len(monthly_data):,} | "
        f"coverage={valid.mean():.4%} | "
        f"unresolved={int((~valid).sum())}"
    )

    del monthly_data, timestamps

timestamp_audit = pd.DataFrame(
    timestamp_audit_results
)

print("\n=== Full Prediction Timestamp Audit ===")

display(timestamp_audit)

assert len(timestamp_audit) == 20
assert timestamp_audit["Records"].sum() == 2_498_494

print("\nFull timestamp audit completed!")

In [ ]:
total_records = int(
    timestamp_audit["Records"].sum()
)

total_valid = int(
    timestamp_audit["ValidTimestamps"].sum()
)

total_dst = int(
    timestamp_audit["UnresolvedDST"].sum()
)

total_2400 = int(
    timestamp_audit["Unresolved2400"].sum()
)

print("=== Timestamp Coverage Summary ===")

print(f"Total records: {total_records:,}")
print(f"Valid timestamps: {total_valid:,}")
print(f"Unresolved DST: {total_dst:,}")
print(f"Unresolved 2400: {total_2400:,}")
print(f"Overall coverage: {total_valid / total_records:.6%}")

assert total_valid + total_dst + total_2400 == total_records

print("\nEarliest cutoff:")
print(timestamp_audit["EarliestPredictionUTC"].min())

print("\nLatest cutoff:")
print(timestamp_audit["LatestPredictionUTC"].max())

In [ ]:
from flight_delay.features.availability import (
    eligible_source_months,
)

earliest_cutoff = timestamp_audit[
    "EarliestPredictionUTC"
].min()

earliest_history = eligible_source_months(
    historical_calendar,
    prediction_at_utc=earliest_cutoff,
    lookback_months=3,
)

print("=== Earliest Historical Coverage ===")
print("Earliest prediction cutoff:", earliest_cutoff)

display(
    earliest_history[
        [
            "report_month",
            "assumed_available_at_utc",
        ]
    ]
)

assert len(earliest_history) == 3

assert earliest_history["report_month"].tolist() == [
    "2022-06",
    "2022-07",
    "2022-08",
]

assert (
    earliest_history["assumed_available_at_utc"]
    <= earliest_cutoff
).all()

print("\nEarliest historical coverage passed!")


## 11. Point-in-Time Historical Window Construction

### Objective

Construct historical feature windows using the latest three eligible BTS reporting months at each source availability boundary.

### Approach

Historical windows are precomputed rather than independently constructed for every target flight.

Each prediction timestamp is matched to the latest window whose availability timestamp is less than or equal to the prediction cutoff.

### Leakage Prevention

- Future reporting months must not contribute to historical features.
- Missing prediction timestamps must not match any historical window.
- Incomplete historical windows are explicitly identified.
- Historical window metadata is retained for auditing.

### Limitation

Point-in-time eligibility is enforced under the assumed 120-day research-proxy policy.

Actual historical publication timestamps and source snapshot versions have not been independently verified.


In [ ]:
from flight_delay.features.historical_lookup import (
    build_history_windows,
    match_history_windows,
)

historical_windows = build_history_windows(
    historical_calendar,
    lookback_months=3,
)

print("=== Historical Windows ===")

display(historical_windows.head(10))

print("Total windows:", len(historical_windows))

print(
    "Complete windows:",
    historical_windows["window_complete"].sum(),
)

print(
    "Incomplete windows:",
    historical_windows["window_complete"].eq(False).sum(),
)

assert len(historical_windows) == 27
assert historical_windows["window_complete"].sum() == 25

assert not historical_windows["window_id"].duplicated().any()

print("\nHistorical window construction passed!")

In [ ]:
boundary = historical_windows.loc[
    historical_windows["window_id"].eq("2022-09"),
    "window_available_at_utc",
].iloc[0]

boundary_examples = pd.Series(
    [
        boundary - pd.Timedelta(seconds=1),
        boundary,
        boundary + pd.Timedelta(seconds=1),
        pd.NaT,
    ],
    dtype="datetime64[ns, UTC]",
    name="prediction_at_utc",
)

boundary_matches = match_history_windows(
    boundary_examples,
    historical_windows,
)

display(
    pd.concat(
        [boundary_examples, boundary_matches],
        axis=1,
    )
)

assert boundary_matches.loc[0, "window_id"] == "2022-08"
assert boundary_matches.loc[1, "window_id"] == "2022-09"
assert boundary_matches.loc[2, "window_id"] == "2022-09"

assert pd.isna(
    boundary_matches.loc[3, "window_id"]
)

assert (
    boundary_matches.loc[3, "history_status"]
    == "unresolved_timestamp"
)

print("\nPoint-in-time boundary checks passed!")

In [ ]:
jan_history_windows = match_history_windows(
    jan_prediction_times["prediction_at_utc"],
    historical_windows,
)

print("=== January 2023 Historical Window Lookup ===")

print("Source flights:", len(jan_raw))
print("Matched records:", len(jan_history_windows))

print("\nHistory status:")
display(
    jan_history_windows["history_status"].value_counts()
)

print("\nHistorical window distribution:")
display(
    jan_history_windows["window_id"].value_counts()
)

assert len(jan_history_windows) == len(jan_raw)

assert jan_history_windows.index.equals(jan_raw.index)

assert jan_history_windows["history_status"].eq("ok").all()

assert jan_history_windows["history_month_count"].eq(3).all()

assert (
    jan_history_windows["window_available_at_utc"]
    <= jan_prediction_times["prediction_at_utc"]
).all()

assert set(jan_history_windows["window_id"]) == {
    "2022-08",
    "2022-09",
}

display(
    pd.concat(
        [
            jan_raw[
                ["FlightDate", "Origin", "CRSDepTime"]
            ],
            jan_prediction_times[
                ["prediction_at_utc"]
            ],
            jan_history_windows,
        ],
        axis=1,
    ).head(10)
)

print("\nJanuary historical window lookup passed!")

In [ ]:
from flight_delay.features.historical_aggregation import (
    COUNT_COLUMNS,
)

origin_snapshot_tables = []
airline_snapshot_tables = []

complete_windows = historical_windows.loc[
    historical_windows["window_complete"]
]

for window in complete_windows.itertuples(index=False):

    source_months = list(window.source_months)

    for stats, keys, tables in [
        (
            historical_origin_stats,
            ["Origin"],
            origin_snapshot_tables,
        ),
        (
            historical_airline_origin_stats,
            ["Origin", "Reporting_Airline"],
            airline_snapshot_tables,
        ),
    ]:

        eligible_stats = stats.loc[
            stats["report_month"].isin(source_months)
        ]

        assert set(eligible_stats["report_month"]) == set(
            source_months
        )

        snapshot = (
            eligible_stats
            .groupby(keys, observed=True)[list(COUNT_COLUMNS)]
            .sum()
            .reset_index()
        )

        snapshot.insert(
            0,
            "window_id",
            window.window_id,
        )

        snapshot["dep_delay_rate"] = (
            snapshot["dep_delay_count"]
            / snapshot["flight_count"]
        )

        snapshot["arr_delay_rate"] = (
            snapshot["arr_delay_count"]
            / snapshot["flight_count"]
        )

        tables.append(snapshot)

origin_snapshots = pd.concat(
    origin_snapshot_tables,
    ignore_index=True,
)

airline_snapshots = pd.concat(
    airline_snapshot_tables,
    ignore_index=True,
)

print("=== Historical Feature Snapshots ===")

print("Origin snapshots:", origin_snapshots.shape)
print("Airline-origin snapshots:", airline_snapshots.shape)

display(origin_snapshots.head(10))
display(airline_snapshots.head(10))

for stats in [origin_snapshots, airline_snapshots]:

    assert stats["flight_count"].gt(0).all()

    assert stats["dep_delay_rate"].between(0, 1).all()
    assert stats["arr_delay_rate"].between(0, 1).all()

assert not origin_snapshots.duplicated(
    ["window_id", "Origin"]
).any()

assert not airline_snapshots.duplicated(
    ["window_id", "Origin", "Reporting_Airline"]
).any()

assert len(origin_snapshots) == 150

print("\nHistorical feature snapshots passed!")


## 12. Historical Feature Join and Fallback Strategy

### Objective

Join point-in-time historical delay statistics to individual target flights.

### Historical Features

- Origin departure delay rate
- Origin outbound arrival delay rate
- Airline-origin departure delay rate
- Airline-origin arrival delay rate

Each rate is calculated from the latest three eligible historical reporting months.

### Fallback Strategy

Airline-origin delay rates are used when at least 100 eligible historical flights are available.

Low-sample and unknown airline-origin groups fall back to the corresponding origin-level rates.

Unavailable historical windows and unresolved prediction timestamps retain missing feature values.

### Quality Controls

- Preserve flight row count and order.
- Enforce many-to-one snapshot joins.
- Retain historical window and fallback metadata.
- Validate historical delay rates.
- Prevent unavailable historical windows from contributing features.

The minimum sample threshold is a predefined engineering heuristic rather than an empirically optimized parameter.

Historical availability remains based on the research-proxy policy.


In [ ]:
from flight_delay.features.historical_features import (
    HISTORICAL_FEATURE_COLUMNS,
    build_historical_features,
)

jan_historical_features = build_historical_features(
    flights=jan_raw,
    prediction_at_utc=jan_prediction_times[
        "prediction_at_utc"
    ],
    windows=historical_windows,
    origin_snapshots=origin_snapshots,
    airline_snapshots=airline_snapshots,
    min_airline_flights=100,
)

print("=== January Historical Features ===")

print("Source flights:", len(jan_raw))
print("Historical feature shape:", jan_historical_features.shape)

display(
    pd.concat(
        [
            jan_raw[
                ["FlightDate", "Origin", "Reporting_Airline"]
            ],
            jan_historical_features[
                list(HISTORICAL_FEATURE_COLUMNS)
            ],
        ],
        axis=1,
    ).head(10)
)

print("\nFallback distribution:")

display(
    jan_historical_features[
        "airline_history_status"
    ].value_counts()
)

print("\nFeature missing values:")

display(
    jan_historical_features[
        list(HISTORICAL_FEATURE_COLUMNS)
    ].isna().sum()
)

print("\nHistorical feature statistics:")

display(
    jan_historical_features[
        list(HISTORICAL_FEATURE_COLUMNS)
    ].describe()
)

In [ ]:
feature_columns = list(HISTORICAL_FEATURE_COLUMNS)

# Verify flight alignment.
assert len(jan_historical_features) == len(jan_raw)

assert jan_historical_features.index.equals(
    jan_raw.index
)

# January has complete historical window coverage.
assert jan_historical_features[
    "history_status"
].eq("ok").all()

# Verify point-in-time eligibility.
assert (
    jan_historical_features["window_available_at_utc"]
    <= jan_prediction_times["prediction_at_utc"]
).all()

# All January origin groups should have historical coverage.
assert jan_historical_features[
    feature_columns
].notna().all().all()

# Verify valid probability ranges.
for column in feature_columns:
    assert jan_historical_features[
        column
    ].between(0, 1).all()

# Verify fallback behavior.
fallback = jan_historical_features[
    "airline_history_status"
].isin([
    "fallback_low_sample",
    "fallback_unknown_group",
])

assert (
    jan_historical_features.loc[
        fallback,
        "airline_origin_dep_delay_rate_3m",
    ]
    == jan_historical_features.loc[
        fallback,
        "origin_dep_delay_rate_3m",
    ]
).all()

assert (
    jan_historical_features.loc[
        fallback,
        "airline_origin_arr_delay_rate_3m",
    ]
    == jan_historical_features.loc[
        fallback,
        "origin_outbound_arr_delay_rate_3m",
    ]
).all()

direct = jan_historical_features[
    "airline_history_status"
].eq("direct")

assert jan_historical_features.loc[
    direct,
    "airline_origin_history_count",
].ge(100).all()

print("=== January Historical Feature Validation ===")

print("Record alignment: Passed")
print("Point-in-time eligibility: Passed")
print("Valid probability range: Passed")
print("Fallback consistency: Passed")

print("Direct matches:", int(direct.sum()))
print("Fallback matches:", int(fallback.sum()))

print("\nJanuary historical feature validation passed!")

In [ ]:
import numpy as np

case_mask = (
    jan_raw["Origin"].eq("ATL")
    & jan_raw["Reporting_Airline"].eq("DL")
    & jan_historical_features["window_id"].eq("2022-08")
)

assert case_mask.any()

case_features = jan_historical_features.loc[
    case_mask,
    list(HISTORICAL_FEATURE_COLUMNS),
].iloc[0]

expected_values = [
    19681 / 80469,
    18276 / 80469,
    12683 / 53533,
    11741 / 53533,
]

np.testing.assert_allclose(
    case_features.to_numpy(dtype=float),
    expected_values,
    rtol=1e-10,
    atol=1e-12,
)

print("=== Manual Feature Verification: DL at ATL ===")

display(case_features)

print("\nManual historical feature verification passed!")

In [ ]:
fallback_examples = pd.DataFrame({
    "Origin": ["ATL", "ATL", "ATL", "ATL", "ATL"],
    "Reporting_Airline": ["DL", "ZZ", "AS", "DL", "DL"],
})

fallback_cutoffs = pd.Series(
    [
        pd.Timestamp("2023-01-10T12:00:00Z"),
        pd.Timestamp("2023-01-10T12:00:00Z"),
        pd.Timestamp("2023-01-10T12:00:00Z"),
        pd.NaT,
        pd.Timestamp("2022-10-01T00:00:00Z"),
    ],
    dtype="datetime64[ns, UTC]",
)

# Use a higher threshold to deliberately trigger low-sample fallback.
fallback_demo = build_historical_features(
    flights=fallback_examples,
    prediction_at_utc=fallback_cutoffs,
    windows=historical_windows,
    origin_snapshots=origin_snapshots,
    airline_snapshots=airline_snapshots,
    min_airline_flights=500,
)

display(
    pd.concat(
        [fallback_examples, fallback_demo],
        axis=1,
    )
)

assert fallback_demo[
    "airline_history_status"
].tolist() == [
    "direct",
    "fallback_unknown_group",
    "fallback_low_sample",
    "history_unavailable",
    "history_unavailable",
]

# Unknown airline and low-sample airline use the origin rates.
for row in [1, 2]:

    assert np.isclose(
        fallback_demo.loc[
            row, "airline_origin_dep_delay_rate_3m"
        ],
        fallback_demo.loc[
            row, "origin_dep_delay_rate_3m"
        ],
    )

    assert np.isclose(
        fallback_demo.loc[
            row, "airline_origin_arr_delay_rate_3m"
        ],
        fallback_demo.loc[
            row, "origin_outbound_arr_delay_rate_3m"
        ],
    )

# Unresolved timestamps and unavailable history stay missing.
assert fallback_demo.loc[
    [3, 4],
    list(HISTORICAL_FEATURE_COLUMNS),
].isna().all().all()

print("\nHistorical feature fallback checks passed!")


## 13. Automated Leakage Tests and Full Historical Feature Generation

### Objective

Validate historical feature generation through automated tests and extend the pipeline to the complete training and validation datasets.

### Validation Scope

- Weighted historical rate calculation
- Source availability boundary handling
- Future-source perturbation tests
- Low-sample and unknown-group fallback
- Missing prediction timestamp handling
- Flight record alignment
- Feature coverage and probability ranges

### Dataset Coverage

- Training: January–December 2023
- Validation: January–August 2024

Historical source eligibility is enforced under the 120-day research-proxy policy.

Calibration, threshold-selection, and final test targets remain excluded.


In [ ]:
from flight_delay.features.historical_aggregation import (
    build_rate_snapshots,
)

new_origin_snapshots = build_rate_snapshots(
    historical_origin_stats,
    historical_windows,
    group_keys=["Origin"],
)

new_airline_snapshots = build_rate_snapshots(
    historical_airline_origin_stats,
    historical_windows,
    group_keys=["Origin", "Reporting_Airline"],
)

for old, new in [
    (origin_snapshots, new_origin_snapshots),
    (airline_snapshots, new_airline_snapshots),
]:
    pd.testing.assert_frame_equal(
        old.reset_index(drop=True),
        new.reset_index(drop=True),
        check_dtype=False,
    )

origin_snapshots = new_origin_snapshots
airline_snapshots = new_airline_snapshots

print("Origin snapshots:", origin_snapshots.shape)
print("Airline snapshots:", airline_snapshots.shape)

print("\nSnapshot regression verification passed!")

In [ ]:
from flight_delay.features.availability import (
    eligible_source_months,
)
from flight_delay.features.historical_features import (
    HISTORICAL_FEATURE_COLUMNS,
    build_historical_features,
)

feature_columns = list(HISTORICAL_FEATURE_COLUMNS)

identity_columns = [
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
]

historical_features_by_month = {}
feature_audit_results = []

for row in target_inventory.itertuples(index=False):

    month = row.report_month

    monthly_data = load_parquet(
        ROOT / "data/processed" / row.processed_filename
    )

    assert len(monthly_data) == row.processed_row_count

    timestamps = build_prediction_timestamps(
        monthly_data
    )

    features = build_historical_features(
        flights=monthly_data,
        prediction_at_utc=timestamps["prediction_at_utc"],
        windows=historical_windows,
        origin_snapshots=origin_snapshots,
        airline_snapshots=airline_snapshots,
        min_airline_flights=100,
    )

    # Check record alignment.
    assert len(features) == len(monthly_data)
    assert features.index.equals(monthly_data.index)

    valid_time = timestamps["timestamp_status"].eq("ok")

    # All resolved development timestamps should have
    # complete historical windows.
    assert features.loc[
        valid_time, "history_status"
    ].eq("ok").all()

    assert (
        features.loc[
            valid_time, "window_available_at_utc"
        ]
        <= timestamps.loc[
            valid_time, "prediction_at_utc"
        ]
    ).all()

    # Independently verify each matched window
    # at its earliest and latest prediction cutoffs.
    audit = features.loc[
        valid_time, ["window_id", "source_months"]
    ].copy()

    audit["prediction_at_utc"] = timestamps.loc[
        valid_time, "prediction_at_utc"
    ]

    for _, group in audit.groupby("window_id"):

        expected_months = group["source_months"].iloc[0]

        for cutoff in [
            group["prediction_at_utc"].min(),
            group["prediction_at_utc"].max(),
        ]:
            eligible = eligible_source_months(
                historical_calendar,
                prediction_at_utc=cutoff,
                lookback_months=3,
            )

            assert tuple(
                eligible["report_month"]
            ) == expected_months

    # Unresolved timestamps must not receive historical rates.
    assert features.loc[
        ~valid_time, feature_columns
    ].isna().all().all()

    for column in feature_columns:
        observed = features[column].dropna()
        assert observed.between(0, 1).all()

    complete_features = features[
        feature_columns
    ].notna().all(axis=1)

    status = features["airline_history_status"]

    feature_audit_results.append({
        "Month": month,
        "Records": len(monthly_data),
        "CompleteFeatures": int(complete_features.sum()),
        "MissingFeatures": int((~complete_features).sum()),
        "DirectMatches": int(status.eq("direct").sum()),
        "LowSampleFallback": int(
            status.eq("fallback_low_sample").sum()
        ),
        "UnknownGroupFallback": int(
            status.eq("fallback_unknown_group").sum()
        ),
        "MissingOriginHistory": int(
            status.eq("missing_origin_history").sum()
        ),
        "UnavailableHistory": int(
            status.eq("history_unavailable").sum()
        ),
    })

    # Keep flight identifiers for reproducible feature joins.
    # Do not include target labels or audit metadata
    # in the historical model feature matrix.
    identity = monthly_data[identity_columns].copy()

    assert not identity.isna().any().any()
    assert not identity.duplicated().any()

    historical_features_by_month[month] = pd.concat(
        [identity, features[feature_columns]],
        axis=1,
    ).reset_index(drop=True)

    print(
        f"{month} | "
        f"records={len(monthly_data):,} | "
        f"coverage={complete_features.mean():.6%} | "
        f"missing={(~complete_features).sum()}"
    )

    del monthly_data, timestamps, features, audit, group

feature_audit = pd.DataFrame(feature_audit_results)

print("\n=== Full Historical Feature Audit ===")

display(feature_audit)

print("\nFull historical feature generation completed!")

In [ ]:
total_records = int(feature_audit["Records"].sum())

complete_records = int(
    feature_audit["CompleteFeatures"].sum()
)

missing_records = int(
    feature_audit["MissingFeatures"].sum()
)

status_columns = [
    "DirectMatches",
    "LowSampleFallback",
    "UnknownGroupFallback",
    "MissingOriginHistory",
    "UnavailableHistory",
]

status_totals = feature_audit[status_columns].sum()

assert total_records == 2_498_494
assert complete_records + missing_records == total_records
assert status_totals.sum() == total_records
assert len(historical_features_by_month) == 20

print("=== Historical Feature Coverage Summary ===")

print(f"Total records: {total_records:,}")
print(f"Complete historical features: {complete_records:,}")
print(f"Missing historical features: {missing_records:,}")
print(f"Coverage: {complete_records / total_records:.6%}")

print("\n=== Fallback Summary ===")
display(status_totals)

print("\n=== Split-Level Feature Coverage ===")

for name, start, end in [
    ("Training", "2023-01", "2023-12"),
    ("Validation", "2024-01", "2024-08"),
]:
    split = feature_audit.loc[
        feature_audit["Month"].between(start, end)
    ]

    records = split["Records"].sum()
    complete = split["CompleteFeatures"].sum()

    print(
        f"{name}: "
        f"records={records:,}, "
        f"complete={complete:,}, "
        f"coverage={complete / records:.6%}"
    )

print("\nHistorical feature coverage audit passed!")


## 14. Historical Feature Dataset Persistence

### Objective

Persist the generated historical features as reusable monthly Parquet datasets.

### Dataset Version

historical_features_v1

### Feature Configuration

- Availability policy: research_proxy
- Assumed publication lag: 120 days
- Historical window: latest three eligible reporting months
- Minimum airline-origin sample count: 100
- Fallback: origin-level historical delay rates

### Storage Design

Each monthly dataset contains flight identity columns and four historical delay features.

Target labels are excluded from the historical feature datasets.

### Reproducibility

Record file checksums, feature configuration, source provenance, row counts, and missing feature counts in a feature manifest.

The calibration, threshold-selection, and final test datasets remain excluded.


In [ ]:
import json

from flight_delay.data.manifest import sha256_file

FEATURE_DIR = (
    ROOT / "data/interim/historical_features_v1"
)

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

feature_file_records = []

for month, frame in sorted(
    historical_features_by_month.items()
):

    expected_columns = (
        identity_columns + feature_columns
    )

    assert frame.columns.tolist() == expected_columns
    assert not frame.duplicated(identity_columns).any()

    path = FEATURE_DIR / f"historical_{month}.parquet"

    temporary_path = path.with_suffix(".tmp.parquet")

    frame.to_parquet(
        temporary_path,
        index=False,
    )

    # Verify Parquet round-trip before replacing the file.
    restored = pd.read_parquet(temporary_path)

    pd.testing.assert_frame_equal(
        frame.reset_index(drop=True),
        restored,
        check_dtype=False,
    )

    temporary_path.replace(path)

    feature_file_records.append({
        "period": month,
        "filename": path.name,
        "row_count": len(frame),
        "column_count": len(frame.columns),
        "missing_feature_rows": int(
            frame[feature_columns].isna().any(axis=1).sum()
        ),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })

    print(
        f"{month} | "
        f"rows={len(frame):,} | "
        f"saved={path.name}"
    )

    del restored

assert len(feature_file_records) == 20

print("\nMonthly historical feature files saved!")

In [ ]:
from datetime import datetime, timezone

source_inventory = []

for row in historical_provenance.itertuples(index=False):

    source_inventory.append({
        "report_month": row.report_month,
        "source_sha256": row.source_sha256,
        "processed_sha256": row.processed_sha256,
        "assumed_available_at_utc": (
            row.assumed_available_at_utc.isoformat()
        ),
    })

audit_path = FEATURE_DIR / "feature_audit.csv"

feature_audit.to_csv(
    audit_path,
    index=False,
)

feature_manifest = {
    "feature_version": "historical_features_v1",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "configuration": {
        "availability_policy": "research_proxy",
        "availability_evidence": "assumed_not_verified",
        "lag_days": 120,
        "lookback_months": 3,
        "min_airline_flights": 100,
        "fallback_strategy": "origin_level",
    },

    "identity_columns": identity_columns,
    "feature_columns": feature_columns,

    "source_inventory": source_inventory,

    "target_coverage": {
        "start": "2023-01",
        "end": "2024-08",
        "months": len(feature_file_records),
        "total_rows": int(
            feature_audit["Records"].sum()
        ),
        "complete_feature_rows": int(
            feature_audit["CompleteFeatures"].sum()
        ),
        "missing_feature_rows": int(
            feature_audit["MissingFeatures"].sum()
        ),
    },

    "audit_file": {
        "filename": audit_path.name,
        "sha256": sha256_file(audit_path),
    },

    "files": feature_file_records,
}

manifest_path = FEATURE_DIR / "feature_manifest.json"

temporary_manifest = manifest_path.with_suffix(".tmp")

with temporary_manifest.open(
    "w", encoding="utf-8"
) as f:
    json.dump(
        feature_manifest,
        f,
        indent=2,
    )

temporary_manifest.replace(manifest_path)

print("=== Historical Feature Manifest ===")

print(json.dumps(
    feature_manifest["configuration"],
    indent=2,
))

print(json.dumps(
    feature_manifest["target_coverage"],
    indent=2,
))

print("\nManifest saved:", manifest_path.relative_to(ROOT))

In [ ]:
with manifest_path.open(encoding="utf-8") as f:
    saved_manifest = json.load(f)

saved_records = 0
saved_missing = 0

expected_months = set(
    pd.period_range(
        "2023-01",
        "2024-08",
        freq="M",
    ).astype(str)
)

assert len(saved_manifest["files"]) == 20

assert {
    record["period"]
    for record in saved_manifest["files"]
} == expected_months

for record in saved_manifest["files"]:

    path = FEATURE_DIR / record["filename"]

    assert path.is_file()

    assert sha256_file(path) == record["sha256"]

    frame = pd.read_parquet(path)

    assert len(frame) == record["row_count"]

    assert frame.columns.tolist() == (
        identity_columns + feature_columns
    )

    assert not frame.duplicated(identity_columns).any()

    missing = int(
        frame[feature_columns].isna().any(axis=1).sum()
    )

    assert missing == record["missing_feature_rows"]

    saved_records += len(frame)
    saved_missing += missing

    del frame

assert saved_records == 2_498_494
assert saved_missing == 2

print("=== Saved Historical Feature Verification ===")

print("Monthly feature files:", len(saved_manifest["files"]))
print(f"Total saved records: {saved_records:,}")
print(f"Missing feature records: {saved_missing:,}")

print("All file checksums: Passed")
print("Flight identity uniqueness: Passed")
print("Feature schema: Passed")

print("\nSaved historical feature verification passed!")

In [ ]:
import numpy as np

from flight_delay.features.build import (
    build_baseline_features,
)

raw = load_parquet(
    ROOT / "data/processed/bts_2023_01_completed.parquet"
)

history = pd.read_parquet(
    FEATURE_DIR / "historical_2023-01.parquet"
)

assert len(raw) == len(history)

assert not raw[identity_columns].duplicated().any()
assert not history[identity_columns].duplicated().any()

assert not raw[identity_columns].isna().any().any()
assert not history[identity_columns].isna().any().any()

# Assign a temporary row ID to preserve the raw data order.
raw_with_order = raw.assign(
    _row_id=np.arange(len(raw))
)

joined = raw_with_order.merge(
    history,
    on=identity_columns,
    how="left",
    validate="one_to_one",
    indicator=True,
    sort=False,
)

assert len(joined) == len(raw)
assert joined["_merge"].eq("both").all()

joined = (
    joined.sort_values("_row_id")
    .reset_index(drop=True)
)

assert np.array_equal(
    joined["_row_id"].to_numpy(),
    np.arange(len(raw)),
)

pd.testing.assert_frame_equal(
    joined[identity_columns],
    raw[identity_columns].reset_index(drop=True),
    check_dtype=False,
)

baseline_input_columns = [
    "FlightDate",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "CRSArrTime",
    "CRSElapsedTime",
    "Distance",
]

X_baseline = build_baseline_features(
    joined[baseline_input_columns]
)

X_baseline = X_baseline.reset_index(drop=True)

X_history = joined[feature_columns].copy()

X_v2a = pd.concat(
    [X_baseline, X_history.iloc[:, :2]],
    axis=1,
)

X_v2b = pd.concat(
    [X_baseline, X_history],
    axis=1,
)

y_departure = joined["DepDel15"].copy()
y_arrival = joined["ArrDel15"].copy()

assert X_baseline.shape == (112_548, 17)
assert X_v2a.shape == (112_548, 19)
assert X_v2b.shape == (112_548, 21)

assert len(y_departure) == len(X_v2b)
assert len(y_arrival) == len(X_v2b)

for target in ["DepDel15", "ArrDel15"]:
    assert target not in X_v2b.columns

print("=== January Model Integration ===")

print("V1 Baseline:", X_baseline.shape)
print("V2-A Origin History:", X_v2a.shape)
print("V2-B Full History:", X_v2b.shape)

print("Departure target:", y_departure.shape)
print("Arrival target:", y_arrival.shape)

display(X_v2b.head())

print("\nModel integration pilot passed!")